# 06 Random Number Sampling

This notebook builds the random-number sampling foundation of the upcoming Monte Carlo engine. Pricing an equity basket autocall — or any other path-dependent product — comes down to simulating a large number of possible futures of the underlyings and averaging the payoff. To do that we need a stream of "random" numbers with very specific properties, and the right way to turn those numbers into the normally-distributed shocks that drive Brownian motion.

We will:

1. Look at three **pseudo-random** generators (Knuth, L'Ecuyer MRG32k3a, Mersenne Twister) and what makes one better than another.
2. Introduce **low-discrepancy** sequences (Halton, Sobol with Joe-Kuo direction numbers), which trade randomness for *coverage* and are the standard tool for high-dimensional pricing.
3. Convert uniform samples on `(0, 1)` into standard-normal samples with five different transforms (Central Limit, Box-Muller, Moro, Acklam, Wichura AS241), and see why some of them must not be combined with Sobol.
4. See the **factory** enforce the legal pairings.
5. Price a European call by Monte Carlo with two complete stacks (textbook PRNG vs production QMC) and watch the QMC error shrink twice as fast.

Each section has a results block followed by an interpretation paragraph. If you are new to Monte Carlo, the interpretations are the most important part — the code is just the bookkeeping.

## Environment setup

The cell below calls `setup_demo_env()` from `examples/_setup.py`, which performs three steps:

1. Adds the project root to `sys.path` so first-party packages (`database`, `schedules`, `market_structures`, `credit`, ...) import without installation.
2. Redirects the global SQLite path to `examples/demo.db` via `set_db_path()`. The production `quant.db` is never read or written by this notebook.
3. Creates the domain tables and seeds the holidays table (USD/EUR/GBP/PLN, years 2000-2100) on first run; subsequent runs detect existing data and skip seeding.

A status line is printed when the cell runs so you can see which path was taken.

In [ ]:
import sys
sys.path.insert(0, "..")

from examples._setup import setup_demo_env
setup_demo_env()

## 1. Pseudo-random generators

When a Monte Carlo simulation asks for "a random number", what we actually need is a stream of values that *look* random to any reasonable statistical test, but that are produced by a deterministic algorithm so the same seed reproduces the same stream byte-for-byte. That is what a **pseudo-random number generator** (PRNG) does: it holds a small piece of internal state, and each call advances the state and emits one number derived from it.

Three properties of a PRNG matter for Monte Carlo work:

- **Period.** Because the state is finite, the sequence must eventually cycle. The *period* is the length of that cycle — the number of values the generator can emit before it starts repeating itself. After the period is exhausted, the next value is identical to the very first, and the simulation effectively starts reusing the same noise. We want the period to be **much, much** larger than the total samples we will ever draw. A pricing run for a 10-asset basket with 1 million paths and 50 observation dates already consumes ~500 million uniforms; over a year of running we might consume `10^15`. Anything with a short period is unusable in practice.
- **Equidistribution.** Over a long enough window, the empirical histogram of outputs should match the target uniform on `(0, 1)`. The **Kolmogorov-Smirnov** (KS) test measures the worst-case gap between the empirical and theoretical CDFs; the **chi-square** test compares histogram bin counts to expected counts. Both report a p-value: roughly speaking, the probability of seeing this much deviation by chance from a *true* uniform sample. P-values above the usual `0.01` cutoff mean the test sees no significant deviation.
- **Independence.** Successive values should be statistically independent. The simplest probe is the **serial correlation at lag `k`**: the Pearson correlation between `x[i]` and `x[i + k]`. For a healthy PRNG these should be very close to zero for every small lag (within `±1/sqrt(N)` of zero). The **lag-1 scatter plot** is the visual version: if dots cluster on lines or curves, the generator has memory.

The three generators we ship illustrate three eras of PRNG design:

| Generator | Period | What that means in practice |
|---|---|---|
| `KnuthSampler` (subtractive `ran3`) | `~ 2**55 ≈ 3.6 × 10^16` | Enough for textbook examples; outclassed on every modern statistical test, so we keep it only as a historical reference. |
| `MRG32k3aSampler` (L'Ecuyer 1999) | `~ 2**191 ≈ 3.1 × 10^57` | Production-grade. Passes the full BigCrush battery. Crucially supports *substreams*: you can leap thousands of substream lengths ahead in O(log n) time, which is how multi-threaded path engines stay reproducible. |
| `MersenneTwisterSampler` (MT19937) | `2**19937 − 1 ≈ 4.3 × 10^6001` | The most widely-shipped PRNG in scientific computing. Astronomical period and very fast bulk generation, but a few BigCrush failures and slow recovery from poorly-chosen seeds. |

In the cell below we pull `30_000` samples from each generator and check three things: KS and chi-square p-values (does the marginal look uniform?), serial correlations at lags 1, 2, 3 (are consecutive draws independent?), and the lag-1 scatter (does it look like noise rather than a pattern?).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from montecarlo import KnuthSampler, MRG32k3aSampler, MersenneTwisterSampler
from montecarlo.diagnostics import chi_square_uniform, ks_uniform, serial_correlation
from montecarlo.plotting import lag_scatter, marginal_histogram

prngs = {
    "Knuth":    KnuthSampler(seed=42),
    "MRG32k3a": MRG32k3aSampler(seed=42),
    "MT19937":  MersenneTwisterSampler(seed=42),
}

n_samples = 30_000

print(f"{'generator':<10}  {'KS stat':>10}  {'KS p-val':>10}  {'chi2 p-val':>11}  {'lag-1':>8}  {'lag-2':>8}  {'lag-3':>8}")
print("-" * 80)

fig, axes = plt.subplots(3, 2, figsize=(11, 11))
for i, (name, sampler) in enumerate(prngs.items()):
    samples = sampler.next_block(n_samples, 1).ravel()
    ks = ks_uniform(samples)
    chi = chi_square_uniform(samples, bins=50)
    corr = serial_correlation(samples, max_lag=3)
    print(f"{name:<10}  {ks.statistic:>10.4f}  {ks.p_value:>10.4f}  {chi.p_value:>11.4f}  {corr[0]:>+8.4f}  {corr[1]:>+8.4f}  {corr[2]:>+8.4f}")
    marginal_histogram(samples, theoretical="uniform", ax=axes[i, 0], title=f"{name} marginal")
    lag_scatter(samples, lag=1, ax=axes[i, 1], title=f"{name} lag-1 scatter")
plt.tight_layout()
plt.show()

**Reading the table.** All three generators show KS and chi-square p-values well above the conventional `0.01` cutoff — the tests see no significant deviation from a true uniform. The lag correlations are all on the order of `1 / sqrt(30_000) ≈ 0.006`, i.e., the natural sampling noise of correlation itself. The horizontal red line in each histogram is the theoretical uniform density of 1, and the bars sit on it. The lag-1 scatter plots are featureless clouds — no diagonals, no clusters, no banding. This is what a well-behaved PRNG looks like: boring. The differences between Knuth, MRG32k3a, and MT19937 do not show up in a sample this small — they emerge in BigCrush, in deep tail behaviour, and over the very long simulations that exhaust short periods.

## 2. Low-discrepancy (quasi-Monte Carlo) sequences

A PRNG aims for *statistical independence* between successive draws. But for Monte Carlo *integration*, independence is not actually what we need — we need the points to **fill the integration region evenly**. A truly independent sequence has clusters and gaps just by chance, and those clusters and gaps slow convergence.

Low-discrepancy sequences make the opposite trade-off: they abandon independence and aim directly at uniform coverage of the unit cube `(0, 1)^d`. They are *not* random — the same sequence comes out every time — but the points are arranged so that they fill the cube with minimal gaps. The cost of "random clustering" is replaced by a deterministic, near-optimal layout. The standard measure is **star discrepancy**: the worst-case difference, over all axis-aligned rectangles, between the fraction of points inside the rectangle and the volume of the rectangle. Smaller star discrepancy means smoother coverage, and smoother coverage translates directly into faster Monte Carlo error decay:

- **PRNG error** decays as `1 / sqrt(N)` — to halve the error you need 4 times the paths.
- **A good low-discrepancy sequence** pushes the decay close to `1 / N` — to halve the error you need only 2 times the paths.

In high dimensions this is a *huge* win. An equity basket autocall on 5 underlyings with 24 quarterly observations already uses 120 dimensions; the QMC speedup is exactly what makes the calculation tractable on a laptop.

We ship two low-discrepancy samplers:

- **`HaltonSampler`** uses the simplest construction: dimension `d` walks through fractions whose digits are written in base `p_d` (the `d`-th prime) and then read in reverse — the *van der Corput* sequence. It is fine for low dimensions but breaks down quickly: pairs of higher-prime coordinates form diagonal stripes that cause systematic bias on any payoff that depends on the joint distribution of those dimensions.
- **`SobolSampler`** uses Joe & Kuo's (2008) primitive-polynomial direction numbers and a gray-code XOR recurrence. The table shipped in `_joe_kuo_data.py` covers 1024 dimensions — more than enough for any reasonable basket autocall — and the construction guarantees well-distributed 2D projections.

The **L2 (centred) discrepancy** below is a closed-form variant of star discrepancy that is much cheaper to compute. Lower is better. The projection grids show the first six coordinates of each sequence plotted pairwise: a healthy QMC sequence fills each 2D panel evenly; a degenerate one shows visible structure (stripes, gaps, clusters).

In [ ]:
from montecarlo import HaltonSampler, SobolSampler
from montecarlo.diagnostics import l2_discrepancy
from montecarlo.plotting import projection_grid

n_qmc = 1024
halton_pts = HaltonSampler(max_dimensions=40).next_block(n_qmc, 40)
sobol_pts  = SobolSampler(max_dimensions=40).next_block(n_qmc, 40)

halton_l2 = l2_discrepancy(halton_pts)
sobol_l2  = l2_discrepancy(sobol_pts)

print(f"L2 discrepancy at N = {n_qmc}, d = 40")
print(f"  Halton:  {halton_l2:.3e}")
print(f"  Sobol :  {sobol_l2:.3e}")
print(f"  ratio :  Halton is {halton_l2 / sobol_l2:.1f}x worse than Sobol")

fig_h = projection_grid(halton_pts, max_dim=6)
fig_h.suptitle("Halton — first 6 dimensions", y=1.02)
plt.show()

fig_s = projection_grid(sobol_pts, max_dim=6)
fig_s.suptitle("Sobol — first 6 dimensions", y=1.02)
plt.show()

**Reading the results.** At `N = 1024` points in `d = 40` dimensions, Sobol's L2 discrepancy is several times smaller than Halton's — the Sobol points cover 40-dimensional space far more evenly. Looking at the first 6 dimensions of each:

- **Sobol**'s off-diagonal panels are visibly uniform clouds — each 2D projection fills its unit square cleanly, and the diagonal histograms are flat.
- **Halton**'s low-dimension panels (dim 0 vs 1) look fine, but as you move toward dim 4 and dim 5 — where the prime bases are larger (`11` and `13`) — the panels start to show coarse banding. With only `1024` points and a coarse base, the sequence has not yet had time to "fill in" the gaps between bands.

Past roughly dimension 10, Halton becomes unsuitable for any payoff that depends on the joint distribution of high-dimension coordinates, which is why Sobol is the production choice for basket autocalls. We keep Halton in the library mostly to make this contrast visible.

### Which Sobol direction integers do we ship?

The Sobol points you just saw are not generated from a single canonical definition — they depend on a table of **direction integers** chosen *separately for each dimension*. Different authors have published different direction-integer tables, and the choice changes how well-distributed the high-dimension projections of the sequence are. This library ships **one** table; QuantLib (and a few other QMC libraries) ship a menu of historical alternatives.

What we ship: the **Joe-Kuo D6** table from `new-joe-kuo-6.21201` (Joe & Kuo, 2008), covering 21 201 dimensions. The vendored data file lives in `montecarlo/uniform/_joe_kuo_data.py`, and we cross-check it bit-for-bit against `ql.SobolRsg.JoeKuoD6` in the validation suite. The "D" in the name refers to the *dimension-stratification depth* the construction is optimised against: higher `D` means projections onto more coordinates simultaneously are well-equidistributed (lower `t`-value), at the cost of fewer total dimensions supported.

The full landscape, for reference — **none of the alternatives are implemented here**, only catalogued so a future caller knows what else exists:

| QuantLib constant | Source | Status here |
|---|---|---|
| `Unit` | trivial (all-ones direction integers) | not shipped |
| `Jaeckel` | Jäckel (2002), *Monte Carlo Methods in Finance* | not shipped |
| `SobolLevitan` | Levitan (1968), corrected by Joe-Kuo | not shipped |
| `SobolLevitanLemieux` | Lemieux modification of the above | not shipped |
| `JoeKuoD5` | Joe & Kuo (2003), ~1 111 dimensions | not shipped |
| **`JoeKuoD6`** | **Joe & Kuo (2008), 21 201 dimensions** | **shipped** |
| `JoeKuoD7` | Joe & Kuo (2010), stricter `t`-value | not shipped |
| `Kuo`, `Kuo2`, `Kuo3` | F. Kuo (solo), alternative variants | not shipped |

Why D6 is the right default for an equity-basket autocall:

- **Plenty of dimensions for any practical payoff.** A monthly autocall on 5 underlyings over 5 years needs `5 × 60 = 300` dimensions; even daily over 5 years is `5 × 1260 = 6 300`, well below D6's 21 201.
- **Good low- and mid-dimension projections.** When we later add Brownian-bridge / PCA re-ordering to allocate the highest-variance time steps to the lowest Sobol coordinates, the 2D and 3D projections of the *first few* coordinates are what carry most of the integration weight. D6's `t`-values are small there.
- **Industry standard.** This is the set used by NAG, Premia, and most finance papers from ~2010 onward, so cross-checking against published benchmarks is straightforward.

`JoeKuoD7` would marginally improve high-dimension discrepancy at the cost of a lower max dimension, which is not a clear win for our use case. Adding another set later is a localised change — vendor its primitive-polynomial table alongside `_joe_kuo_data.py`, plumb a `direction_set` parameter through `SobolSampler`, add the matching cross-validation test. The gray-code recurrence itself is set-agnostic.

## 3. Transforming `U(0, 1)` to `N(0, 1)`

Most financial models — Black-Scholes, Heston, local-volatility, the basket autocall we are building — drive the underlyings with **Brownian motion**, whose increments are normally distributed. So the random numbers a pricing engine actually consumes are *standard normals* `Z ~ N(0, 1)`, not uniforms. The job of this layer is to convert each uniform `u in (0, 1)` from the sampler into one standard-normal `z`.

The cleanest way to do this is **inverse-CDF sampling**: take `z = Φ^{-1}(u)`, where `Φ` is the standard normal cumulative distribution function and `Φ^{-1}` is its inverse. Geometrically: `Φ` maps `(-∞, +∞)` onto `(0, 1)`; running it backwards takes a uniform `u in (0, 1)` and stretches it into a value on the real line, mapping the deep tails of the uniform (near `0` and `1`) into the deep tails of the normal (large `|z|`). Because `Φ^{-1}` has no elementary closed form, three rational-polynomial approximations are common:

- **`MoroTransform`** (Moro, 1995) — about `3 × 10^{-9}` error in the body, but degrades into the deep tail (around `5 × 10^{-3}` at `Φ^{-1}(10^{-15})`). The most-cited classical approximation in option-pricing textbooks.
- **`AcklamTransform`** (Acklam, 2003) — about `10^{-8}` error across the full range, including the deep tail. A good general-purpose default.
- **`WichuraAS241Transform`** (Wichura, 1988, *Applied Statistics* AS241) — machine precision (`~10^{-15}`) everywhere. This is the algorithm behind R's `qnorm`. Use it when the price depends on deep-tail accuracy: autocall barriers, deep OTM, capital-protected products.

Two other transforms exist for historical / pedagogical reasons:

- **`BoxMullerTransform`** does not use `Φ^{-1}` at all. Given two uniforms `(U1, U2)` it returns `(sqrt(-2 ln U1) cos(2π U2), sqrt(-2 ln U1) sin(2π U2))`. The result is *exactly* `N(0, 1)` with no rational approximation, but the wrap through `cos / sin` is non-monotonic in `U2`, which destroys the equidistribution structure of a low-discrepancy sampler. Box-Muller is therefore safe with PRNGs but **not** with Sobol or Halton.
- **`CLTTransform`** sums 12 uniforms and subtracts 6, leaning on the central limit theorem. Mean and variance come out correct, but the result is hard-bounded in `[-6, +6]` — the tails beyond `±6 σ` are missing entirely. This is fatal for any tail-sensitive product. The library issues a `UserWarning` whenever this transform is constructed.

First we measure raw accuracy: for each inverse-CDF transform, compare its output to `scipy.special.ndtri` (a high-precision reference) over two grids — the central body `(0.025, 0.975)` and the deep tail (uniforms as close as `10^{-15}` to the endpoints).

In [ ]:
from scipy.special import ndtri

from montecarlo import (
    AcklamTransform,
    BoxMullerTransform,
    CLTTransform,
    MoroTransform,
    WichuraAS241Transform,
    make_normal_sampler,
)
from montecarlo.diagnostics import moments, tail_fractions
from montecarlo.plotting import qq_normal

inverse_cdfs = {
    "Moro":          MoroTransform(),
    "Acklam":        AcklamTransform(),
    "Wichura AS241": WichuraAS241Transform(),
}

central_grid = np.linspace(0.025, 0.975, 5001)[None, :]
tail_grid = np.concatenate(
    [np.logspace(-15, -6, 1000), 1.0 - np.logspace(-15, -6, 1000)]
)[None, :]

print(f"{'transform':<15}  {'central max |err|':>20}  {'deep-tail max |err|':>22}")
print("-" * 63)
for name, t in inverse_cdfs.items():
    central_err = float(np.abs(t.transform(central_grid) - ndtri(central_grid)).max())
    tail_err = float(np.abs(t.transform(tail_grid) - ndtri(tail_grid)).max())
    print(f"{name:<15}  {central_err:>20.3e}  {tail_err:>22.3e}")

**Reading the table.** All three transforms are accurate to about `10^{-9}` in the central body of the normal — that is roughly one unit in the last place of a double-precision float, indistinguishable from exact for any practical Monte Carlo. The deep-tail column is what separates them:

- **Wichura AS241** holds machine precision all the way out to `Φ^{-1}(10^{-15}) ≈ -8 σ`.
- **Acklam** loses about six digits in the deep tail but still finishes at `10^{-9}` — for almost any product, well below the price's last meaningful digit.
- **Moro** starts to lose precision past about `Φ^{-1}(10^{-6}) ≈ -4.75 σ` and drifts to ~`5 × 10^{-3}` (half a percentage point of the normal quantile) by `-8 σ`.

For a one-year ATM European call this hardly matters: the payoff is dominated by the central region. For an autocall with a `60 %` downside barrier (around `-3.5 σ` of cumulative return), it begins to matter. For capital-protected products with very deep out-of-the-money knock-ins, it matters a lot, which is why our library default is Wichura AS241.

Raw accuracy on the function `Φ^{-1}` is half the story. The other half is what happens when we run each transform on a finite sample of `100_000` uniforms from MT19937 and look at what comes out: do the first four moments match `N(0, 1)`, and — most importantly — does each transform reproduce the *frequency* of tail events? The empirical fraction of `|Z| > k σ` should match the theoretical value `2 · (1 − Φ(k))` for `k = 2, 3, 4`. CLT, with its hard `±6 σ` cap, is where this test bites.

CLT's constructor would normally emit a `UserWarning`; we silence it in the cell because the whole point here is to *show* the problem rather than be reminded of it.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    clt = CLTTransform()

all_transforms = {
    "CLT":           clt,
    "Box-Muller":    BoxMullerTransform(),
    "Moro":          MoroTransform(),
    "Acklam":        AcklamTransform(),
    "Wichura AS241": WichuraAS241Transform(),
}

n_paths = 100_000
fig, axes = plt.subplots(len(all_transforms), 2, figsize=(11, 3.5 * len(all_transforms)))

txt = ''
results_z = {}
for i, (name, transform) in enumerate(all_transforms.items()):
    ns = make_normal_sampler(MersenneTwisterSampler(seed=2026), transform)
    z = ns.next_block(n_paths, 1).ravel()
    results_z[name] = z
    marginal_histogram(z, theoretical="normal", ax=axes[i, 0], title=f"{name} marginal")
    qq_normal(z, ax=axes[i, 1], title=f"{name} Q-Q vs N(0, 1)")
    m = moments(z)
    txt += f"{name:<15}  {m['mean']:>+8.4f}  {m['variance']:>9.4f}  {m['skewness']:>+9.4f}  {m['excess_kurtosis']:>+11.4f}\n"

print(f"{'transform':<15}  {'mean':>8}  {'variance':>9}  {'skewness':>9}  {'excess kurt':>11}")
print("-" * 60)
print(f"{'theoretical':<15}  {0.0:>+8.4f}  {1.0:>9.4f}  {0.0:>+9.4f}  {0.0:>+11.4f}")
print(txt)

plt.tight_layout()
plt.show()

print()
theo = tail_fractions(np.array([0.0]), sigmas=(2, 3, 4))
print(f"{'transform':<15}  {'P(|Z|>2)':>10}  {'P(|Z|>3)':>10}  {'P(|Z|>4)':>10}")
print("-" * 52)
print(f"{'theoretical':<15}  {theo[2.0]['theoretical']:>10.5f}  {theo[3.0]['theoretical']:>10.5f}  {theo[4.0]['theoretical']:>10.6f}")
for name, z in results_z.items():
    tails = tail_fractions(z, sigmas=(2, 3, 4))
    print(f"{name:<15}  {tails[2.0]['empirical']:>10.5f}  {tails[3.0]['empirical']:>10.5f}  {tails[4.0]['empirical']:>10.6f}")

**Reading the moments table.** All five transforms match the theoretical mean (0), variance (1) and skewness (0) to within sampling noise — at `N = 100_000` the standard error of the sample mean of `N(0, 1)` is `1/sqrt(100_000) ≈ 0.003`, so any value within roughly `±0.01` is consistent with exact. The story lives in the **excess kurtosis** column. A true normal has excess kurtosis 0; CLT shows a small but systematically negative value (`≈ −1/60`), the signature of a *platykurtic* — flat-topped, light-tailed — distribution.

**Reading the tail-fractions table.** Excess kurtosis sounds abstract; the tail-fractions table is where it bites. The theoretical `P(|Z| > 4 σ)` is `~6.3 × 10^{−5}`, i.e., 6 to 7 occurrences per 100 000 paths. The four good transforms all reproduce this. CLT's `P(|Z| > 4)` is **zero**: a sum of 12 uniforms minus 6 is bounded by `±6` exactly, but reaching `|Z| > 4` requires almost all 12 uniforms to be near `1` or near `0`, which essentially never happens at this sample size. For a barrier or autocall payoff "never reaches the tail" means "never gets hit" means "systematically underpriced downside risk". This is *exactly* why CLT is shipped only as a teaching example.

**Reading the Q-Q plots.** A Q-Q plot sorts the sample and plots the i-th smallest value against the theoretical i-th quantile of `N(0, 1)`. A perfect match traces the red diagonal. The CLT plot pinches off cleanly at `±6` — at both ends the points peel off the line and curve back, because the empirical distribution simply cannot go further. The four good transforms all run straight from one end of the diagonal to the other, with the natural sampling jitter only near the extreme corners.

## 4. The QMC / inversion pairing rule

We saw in §2 that Sobol points are carefully arranged to fill `(0, 1)^d` evenly. We saw in §3 that Box-Muller maps `(U1, U2)` through the trigonometric wrap `(cos 2π U2, sin 2π U2)`. Putting them together is a *correctness bug*: the wrap is non-monotonic in `U2`, so two Sobol points that were close in the unit cube can map to two normals that are far apart, and vice versa. The carefully chosen Sobol layout is shuffled into something that is no longer low-discrepancy on the output side, and the QMC convergence advantage evaporates (Boyle, Broadie, Glasserman, 1997, *Monte Carlo Methods for Security Pricing*). The same problem applies to CLT, plus the tail-truncation issue from §3.

This is not a soft preference — it is a correctness rule. `make_normal_sampler(sampler, transform)` is the only documented way to compose a uniform sampler with a transform, and it refuses the two illegal combinations at construction time. The cell below builds every (sampler, transform) pair we ship and prints the outcome.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)

    print(f"{'sampler':<18}  {'transform':<15}  outcome")
    print("-" * 65)
    for sampler_name, sampler_factory in [
        ("MRG32k3a (PRNG)", lambda: MRG32k3aSampler(seed=1)),
        ("Sobol (QMC)",     lambda: SobolSampler(max_dimensions=2)),
    ]:
        for tname, tfact in [
            ("Box-Muller",    BoxMullerTransform),
            ("CLT",           CLTTransform),
            ("Moro",          MoroTransform),
            ("Acklam",        AcklamTransform),
            ("Wichura AS241", WichuraAS241Transform),
        ]:
            try:
                make_normal_sampler(sampler_factory(), tfact())
                outcome = "accepted"
            except ValueError:
                outcome = "REJECTED (not QMC-safe)"
            print(f"{sampler_name:<18}  {tname:<15}  {outcome}")
        print()

**Reading the table.** Every transform composes with the PRNG: with statistically independent uniforms, the trigonometric wrap and the `±6 σ` truncation are statistically fine (CLT is still a bad idea for tail-sensitive products, but it is at least *legal*). With Sobol, only the three inverse-CDF transforms are accepted; Box-Muller and CLT raise `ValueError` at construction with a message explaining why. There is no way to obtain a `NormalSampler` instance that combines Sobol with Box-Muller without bypassing the factory entirely — the safety is at the type-construction level, not a runtime check.

## 5. End-to-end: Black-Scholes convergence

We now stop staring at individual layers and put the full stack to work. The European call option is the standard reference because it has a **closed-form price** (Black-Scholes), so any Monte Carlo estimator can be measured against the truth.

**The instrument.** A 1-year European call on an underlying with spot `S0 = 100`, strike `K = 100`, volatility `σ = 20 %`, risk-free rate `r = 5 %`. The Black-Scholes price is about `10.45`. The Monte Carlo estimator simulates the terminal value

```
S_T  =  S0 · exp((r − ½ σ²) T  +  σ √T · Z)        with   Z ~ N(0, 1)
```

then averages the discounted payoff `exp(−r T) · max(S_T − K, 0)` across many paths. The price is the average; the *error* is how far that average lands from the closed-form value.

**Two complete stacks compared.**

- **MT19937 + Box-Muller** — the textbook PRNG path. We expect the absolute pricing error to decay as `1 / sqrt(N)`.
- **Sobol + Wichura AS241** — the QMC stack this library promotes for tail-sensitive products. We expect the error to decay close to `1 / N` (up to log factors).

**What "convergence" means here.** As we run the MC with more paths `N`, the absolute pricing error `|MC price − BS price|` should shrink. The question is *how fast*.

| Method | Convergence law | "To halve the error..." | "Each time N doubles, error should multiply by..." |
|---|---|---|---|
| PRNG (MT19937 + Box-Muller) | `error ≈ C / sqrt(N)` | ...you need 4× as many paths. | `1 / sqrt(2) ≈ 0.707` |
| QMC (Sobol + AS241) | `error ≈ C / N` (up to log factors) | ...you need 2× as many paths. | `1 / 2 = 0.500` |

So if PRNG gives an error of `0.10` at `N = 1_000`, we expect roughly `0.05` at `N = 4_000` and `0.025` at `N = 16_000`. QMC, starting from the same `0.10`, would give `0.05` at `N = 2_000` and `0.025` at `N = 4_000` — that ratio compounds quickly.

The cell below runs `N = 256, 512, 1024, …, 65_536` for both stacks. For each `N` it prints the actual MC price, the absolute error, and the **error ratio versus the previous `N`** (which should approach `0.707` for PRNG and `0.500` for QMC). The final plot puts both error curves on log-log axes, with two grey reference lines showing the theoretical `1 / sqrt(N)` and `1 / N` rates passing through the leftmost point of each curve.

In [ ]:
from montecarlo.diagnostics.integration import bs_call_price_mc

bs_params = dict(spot=100.0, strike=100.0, rate=0.05, sigma=0.2, maturity=1.0)
benchmark = bs_call_price_mc(
    make_normal_sampler(MRG32k3aSampler(seed=0), WichuraAS241Transform()),
    n_paths=1_000_000,
    **bs_params,
).benchmark

print("Instrument:  1Y European call,  S0 = 100,  K = 100,  r = 5%,  sigma = 20%")
print(f"Black-Scholes closed-form price: {benchmark:.6f}")
print()

path_counts = np.array([256, 512, 1024, 2048, 4096, 8192, 16_384, 32_768, 65_536])
prices_prng = []
prices_sobol = []
for n in path_counts:
    prng_ns = make_normal_sampler(MersenneTwisterSampler(seed=int(n)), BoxMullerTransform())
    prices_prng.append(bs_call_price_mc(prng_ns, n_paths=int(n), **bs_params).estimate)
    sobol_ns = make_normal_sampler(SobolSampler(max_dimensions=2, skip=int(n)), WichuraAS241Transform())
    prices_sobol.append(bs_call_price_mc(sobol_ns, n_paths=int(n), **bs_params).estimate)

prices_prng = np.array(prices_prng)
prices_sobol = np.array(prices_sobol)
err_prng = np.abs(prices_prng - benchmark)
err_sobol = np.abs(prices_sobol - benchmark)

print(f"{'N':>8}  | {'PRNG price':>11}  {'|err|':>8}  {'err/prev':>9}  | {'QMC price':>10}  {'|err|':>8}  {'err/prev':>9}")
print("-" * 92)
for i, n in enumerate(path_counts):
    if i == 0:
        ratio_prng_str = "       -"
        ratio_qmc_str  = "       -"
    else:
        ratio_prng_str = f"{err_prng[i] / err_prng[i-1]:>8.3f}"
        ratio_qmc_str  = f"{err_sobol[i] / err_sobol[i-1]:>8.3f}"
    print(f"{n:>8}  | {prices_prng[i]:>11.6f}  {err_prng[i]:>8.4f}  {ratio_prng_str:>9}  | {prices_sobol[i]:>10.6f}  {err_sobol[i]:>8.4f}  {ratio_qmc_str:>9}")

print()
mean_ratio_prng = float(np.exp(np.mean(np.log(err_prng[1:] / err_prng[:-1]))))
mean_ratio_qmc  = float(np.exp(np.mean(np.log(err_sobol[1:] / err_sobol[:-1]))))
print(f"Geometric-mean error ratio per doubling of N:")
print(f"  PRNG : {mean_ratio_prng:.3f}   (theoretical 1 / sqrt(2) = 0.707)")
print(f"  QMC  : {mean_ratio_qmc:.3f}   (theoretical 1 / 2          = 0.500)")

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.loglog(path_counts, err_prng,  marker="o", color="C0", label="MT19937 + Box-Muller  (PRNG)")
ax.loglog(path_counts, err_sobol, marker="s", color="C1", label="Sobol + Wichura AS241  (QMC)")
ref_prng_curve  = err_prng[0]  * np.sqrt(path_counts[0] / path_counts)
ref_sobol_curve = err_sobol[0] * (path_counts[0] / path_counts)
ax.loglog(path_counts, ref_prng_curve,  "--", color="C0", alpha=0.35, label="reference slope: 1 / sqrt(N)")
ax.loglog(path_counts, ref_sobol_curve, "--", color="C1", alpha=0.35, label="reference slope: 1 / N")
ax.set_xlabel("number of paths  N")
ax.set_ylabel("absolute pricing error  |MC − BS|")
ax.set_title("European call: pricing error vs path count (log-log)")
ax.grid(True, which="both", linestyle=":", alpha=0.5)
ax.legend(loc="lower left")
plt.show()

**Reading the table.** Each row corresponds to a doubling of `N`. Compare the two `err/prev` columns:

- **PRNG (`err/prev`)** fluctuates around `0.7`. Some doublings are lucky (the new MC sample averages out the previous error and the ratio is `0.3` or `0.4`); some doublings are unlucky and the ratio is `1.2`. The *geometric-mean* ratio over all doublings, printed below the table, is what should approach `1 / sqrt(2) ≈ 0.707`. The fluctuation is real and unavoidable: PRNG MC is a random algorithm, so single-doubling ratios are noisy.
- **QMC (`err/prev`)** sits much closer to `0.5`. The Sobol sequence is deterministic, so there is none of PRNG's lottery-ticket variance: each new doubling cuts the error roughly in half.

By `N = 65_536`, the QMC error is on the order of a fraction of a cent on a price of about `10.45`, while the PRNG error is still several cents. The QMC stack is delivering the same precision with roughly an order of magnitude fewer paths.

**Reading the plot.** Both axes are logarithmic. On a log-log plot a power law `y = C · N^{−p}` becomes a straight line with slope `−p`. The two grey dashed lines have slopes `−1/2` (PRNG-rate, blue dashes) and `−1` (QMC-rate, orange dashes). The blue PRNG curve hugs its reference line — sometimes a bit above, sometimes a bit below, because PRNG MC is noisy — but its long-run slope is clearly `−1/2`. The orange QMC curve hugs its reference line, twice as steep. **That doubled slope is the entire reason we ship Sobol.** It means that for any fixed compute budget Sobol delivers a measurably better price; equivalently, to gain *one more digit* of precision the PRNG path needs `100×` more paths while the QMC path needs only `10×`.

(In a full basket-autocall pricer, the dimension grows from 1 to ~`N_assets × N_observations`, and we will also add Brownian Bridge + PCA factor ordering so the highest-variance dimensions land on the lowest Sobol coordinates. That further compresses the *effective* dimension Sobol has to integrate over and is how QMC stays useful in the hundreds-of-dimensions regime.)

## Summary

We toured the sampling layer of the Monte Carlo engine:

1. **PRNGs** (Knuth, MRG32k3a, MT19937) produce streams that are statistically uniform and independent over realistic sample sizes. The differences between them matter for very long simulations, parallel reproducibility, and BigCrush-level statistical scrutiny.
2. **QMC** sequences (Halton, Sobol) replace independence with deterministic equidistribution, which buys faster convergence at the cost of strict pairing rules with the downstream transform.
3. **Normal transforms** convert uniforms into the normals that Brownian motion needs. The three inverse-CDF transforms (Moro, Acklam, Wichura AS241) span the accuracy-cost trade-off; Box-Muller is the elegant exact alternative that only works with PRNGs; CLT is shipped as a cautionary example.
4. **The factory** enforces the QMC/inversion pairing rule at construction time so a Sobol + Box-Muller error cannot escape into a pricing run.
5. **End-to-end**, the Sobol + Wichura AS241 stack delivers `1 / N` convergence on the Black-Scholes call price — measurably better than the `1 / sqrt(N)` rate of the textbook PRNG stack, and the right default for the basket-autocall pricer being built on top of this layer.

The dimension-aware `next_block(n_paths, n_dimensions)` interface seen throughout will be reused unchanged in the next steps (simulation grid, Brownian Bridge, correlated basket draws, variance reduction).